# Intervention Effectiveness Prediction
Predicting which interventions are most effective for individual residents.

## 1. Problem Framing
Different residents respond differently to available programs (counseling, job training, life skills, substance-use treatment, etc.). This model predicts the probability that a given intervention leads to a positive outcome (defined as reintegration within 6 months) for a specific resident profile.

**Approach:** Multi-class / multi-label recommendation framed as a binary outcome model per intervention type, combined with a causal uplift analysis to distinguish correlation from genuine treatment effect.

**Target variable:** `positive_outcome` — binary (1 = achieved stable housing within 6 months after intervention, 0 = did not).

**Success metric:** ROC-AUC ≥ 0.75 per intervention type; statistically significant average treatment effect (ATE) for the recommended intervention vs. control.

**Stakeholders:** Program directors, case managers.

## 2. Data Acquisition & Preparation

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
n = 900

interventions = ['Counseling', 'Job Training', 'Life Skills', 'Substance Use Treatment', 'Education']

df = pd.DataFrame({
    'resident_id': range(1, n + 1),
    'age': np.random.randint(18, 65, size=n),
    'gender': np.random.choice(['M', 'F', 'Other'], size=n, p=[0.5, 0.45, 0.05]),
    'mental_health_score': np.random.randint(1, 11, size=n),
    'substance_use_severity': np.random.randint(0, 11, size=n),
    'prior_employment_years': np.random.uniform(0, 20, size=n),
    'education_level': np.random.randint(0, 5, size=n),  # 0=none, 4=college+
    'social_support_score': np.random.randint(0, 11, size=n),
    'assigned_intervention': np.random.choice(interventions, size=n),
    'intervention_duration_weeks': np.random.randint(4, 52, size=n),
    'compliance_rate': np.random.uniform(0, 1, size=n),
})

df = pd.get_dummies(df, columns=['gender', 'assigned_intervention'], drop_first=True)

# Simulate outcome
intervention_bonus = 0.0
if 'assigned_intervention_Job Training' in df.columns:
    intervention_bonus += 0.10 * df['assigned_intervention_Job Training']
if 'assigned_intervention_Counseling' in df.columns:
    intervention_bonus += 0.08 * df['assigned_intervention_Counseling']

outcome_prob = (
    0.15
    + 0.015 * df['mental_health_score']
    - 0.012 * df['substance_use_severity']
    + 0.01 * df['social_support_score']
    + 0.008 * df['prior_employment_years']
    + 0.08 * df['compliance_rate']
    + intervention_bonus
).clip(0.05, 0.92)
df['positive_outcome'] = np.random.binomial(1, outcome_prob)

print(df.shape)
print(df['positive_outcome'].value_counts())
df.head()

In [ ]:
feature_cols = [
    'age', 'mental_health_score', 'substance_use_severity',
    'prior_employment_years', 'education_level', 'social_support_score',
    'intervention_duration_weeks', 'compliance_rate'
] + [c for c in df.columns if c.startswith('gender_') or c.startswith('assigned_intervention_')]

X = df[feature_cols]
y = df['positive_outcome']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 3. Exploration

In [ ]:
import matplotlib.pyplot as plt

# Outcome rate by intervention (reconstruct original column from dummies)
intervention_cols = [c for c in df.columns if c.startswith('assigned_intervention_')]
df_plot = df.copy()
df_plot['intervention'] = 'Reference'
for col in intervention_cols:
    df_plot.loc[df_plot[col] == 1, 'intervention'] = col.replace('assigned_intervention_', '')

outcome_by_intervention = df_plot.groupby('intervention')['positive_outcome'].mean().sort_values()
outcome_by_intervention.plot(kind='barh', color='mediumseagreen', figsize=(8, 4))
plt.title('Positive Outcome Rate by Intervention Type')
plt.xlabel('Proportion with Positive Outcome')
plt.tight_layout()
plt.show()

# Compliance vs outcome
fig, ax = plt.subplots(figsize=(6, 4))
for outcome, label, color in [(0, 'No outcome', 'tomato'), (1, 'Positive outcome', 'steelblue')]:
    sub = df[df['positive_outcome'] == outcome]
    ax.scatter(sub['compliance_rate'], sub['mental_health_score'],
               alpha=0.3, s=10, label=label, color=color)
ax.set_xlabel('Compliance Rate')
ax.set_ylabel('Mental Health Score')
ax.set_title('Compliance vs Mental Health by Outcome')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Modeling
### 4a. Predictive Model — Random Forest Classifier
### 4b. Causal / Explanatory Model — statsmodels Logistic Regression with Average Treatment Effects

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# --- 4a: Predictive model ---
rf = RandomForestClassifier(n_estimators=200, max_depth=7, random_state=42, class_weight='balanced')
rf.fit(X_train, y_train)

y_pred_proba = rf.predict_proba(X_test)[:, 1]
y_pred = rf.predict(X_test)
print("Random Forest training complete.")

In [ ]:
# --- 4b: Causal / Explanatory model ---
# Average Treatment Effect (ATE) estimated via propensity-score-weighted Logistic Regression
try:
    import statsmodels.api as sm

    # Use Job Training as the treatment variable
    treatment_col = 'assigned_intervention_Job Training'
    if treatment_col not in X_train.columns:
        print(f"Column '{treatment_col}' not found. Skipping ATE analysis.")
    else:
        # Step 1: Estimate propensity scores
        confounders = [
            'age', 'mental_health_score', 'substance_use_severity',
            'prior_employment_years', 'education_level', 'social_support_score'
        ]
        X_ps = sm.add_constant(X_train[confounders])
        T = X_train[treatment_col].astype(float).reset_index(drop=True)
        ps_model = sm.Logit(T, X_ps.reset_index(drop=True)).fit(disp=False)
        propensity_scores = ps_model.predict(X_ps)

        # Step 2: IPW weights
        weights = np.where(T == 1, 1 / propensity_scores.clip(0.05, 0.95),
                           1 / (1 - propensity_scores.clip(0.05, 0.95)))

        # Step 3: Outcome model with IPW
        X_out = sm.add_constant(X_train.reset_index(drop=True))
        outcome_model = sm.WLS(
            y_train.reset_index(drop=True).astype(float),
            X_out,
            weights=weights
        ).fit()

        ate = outcome_model.params.get(treatment_col, float('nan'))
        pval = outcome_model.pvalues.get(treatment_col, float('nan'))
        print(f"Estimated ATE (Job Training): {ate:.4f}  (p = {pval:.4f})")
        print("Interpretation: Positive ATE implies Job Training increases positive outcome probability.")
except ImportError:
    print("statsmodels not installed — skipping. Install with: pip install statsmodels")

## 5. Evaluation

In [ ]:
from sklearn.metrics import (
    classification_report, roc_auc_score,
    RocCurveDisplay, ConfusionMatrixDisplay
)

print("ROC-AUC:", round(roc_auc_score(y_test, y_pred_proba), 4))
print()
print(classification_report(y_test, y_pred, target_names=['No Outcome', 'Positive Outcome']))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
RocCurveDisplay.from_predictions(y_test, y_pred_proba, ax=axes[0], name='Random Forest')
axes[0].set_title('ROC Curve — Intervention Effectiveness')
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=['No Outcome', 'Positive Outcome'], ax=axes[1]
)
axes[1].set_title('Confusion Matrix')
plt.tight_layout()
plt.show()

## 6. Feature Selection

In [ ]:
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)

plt.figure(figsize=(10, 4))
importances.head(12).plot(kind='bar', color='mediumseagreen')
plt.title('Random Forest Feature Importances — Intervention Effectiveness')
plt.ylabel('Importance')
plt.tight_layout()
plt.show()

selected_features = importances[importances >= importances.mean()].index.tolist()
print("Selected features:", selected_features)

## 7. Deployment

In [ ]:
import pickle, os

os.makedirs('models', exist_ok=True)

X_train_sel = X_train[selected_features]
X_test_sel = X_test[selected_features]

rf_final = RandomForestClassifier(n_estimators=200, max_depth=7, random_state=42, class_weight='balanced')
rf_final.fit(X_train_sel, y_train)

with open('models/intervention_effectiveness_model.pkl', 'wb') as f:
    pickle.dump({'model': rf_final, 'features': selected_features}, f)

print("Model saved to models/intervention_effectiveness_model.pkl")

# Predict which intervention is best for each resident
# In production: run inference for each intervention scenario per resident
sample = X_test_sel.iloc[:5].copy()
sample['outcome_probability'] = rf_final.predict_proba(sample)[:, 1]
print(sample[['outcome_probability']])